# The AI Boom in Numbers

It feels like a new AI model comes out every week. Before coming to SIPA, I cofounded an AI startup. On the business side, I noticed that the conversation kept shifting: every few months there was a new model that customers and investors were excited about. I wanted to know whether this "AI boom" actually shows up in the data, or whether it just feels that way from inside the industry.

To find out, I used Epoch AI's [Notable AI Models](https://epoch.ai/data/ai-models) dataset. Epoch AI is a research nonprofit that tracks the development of artificial intelligence. Each row is one AI model that Epoch considers "notable" — for example, because it was state of the art, widely cited, or historically important. The list goes all the way back to the 1950s.

My question is simple: **when were most notable AI models released?**

## Loading the data

I'm reading the data directly from Epoch AI's website, so this notebook always uses their latest version. Then I check how many models (rows) and pieces of information (columns) it contains.

In [27]:
# data loading code with help from Claude
import pandas as pd

url = "https://epoch.ai/data/notable_ai_models.csv"
models = pd.read_csv(url)

print(models.shape)
models.head()

(1092, 47)


,Model,Organization,Publication date,Domain,Task,Parameters,Parameters notes,Training compute (FLOP),Training compute notes,Training dataset,...,Training code accessibility,Inference code accessibility,Accessibility notes,Numerical format,Frontier model,Hardware acquisition cost,Hardware utilization (HFU),Training compute cost (cloud),Training compute cost (upfront),Open model weights?
0,Gemini 4 Carbon,Google DeepMind,2026-10-28,"Language,Multimodal,Vision","Language modeling/generation,Image Understandi...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Unreleased as of October 28 2026,NaN,NaN,NaN,NaN,NaN,NaN,No
1,MAI-Base-2,Microsoft,2026-10-15,"Language,Vision,Multimodal",Language modeling/generation,2.800000e+12,"""80 billion active and 2.8 trillion total para...",2.400000e+25,6 * 80e9 active parameters * 50e12 tokens = 2...,NaN,...,Unreleased,NaN,NaN,FP8,NaN,6.561456e+08,NaN,NaN,NaN,No
2,Grok Voice 3.0,xAI,2026-10-14,"Language,Audio,Speech",Speech-to-speech,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
3,Muse Wonder Full,Meta AI,2026-10-12,"Language,Vision,Video,Multimodal","Language modeling/generation,Question answerin...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No
4,Muse Wonder,Meta AI,2026-10-12,"Language,Vision,Video,Multimodal","Language modeling/generation,Question answerin...",NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,No


Each model has a full publication date, but I only need the year. I convert the dates and pull out the year into a new column, then check that no years are missing.

In [28]:
models["Publication date"] = pd.to_datetime(models["Publication date"])
models["Year"] = models["Publication date"].dt.year

models["Year"].isna().sum()

np.int64(0)

## When were AI models released?

Using pandas, I calculate the mean, median, and mode of the release year.

In [29]:
year_mean = models["Year"].mean()
year_median = models["Year"].median()
year_mode = models["Year"].mode()[0]

print("Mean:", year_mean)
print("Median:", year_median)
print("Mode:", year_mode)

Mean: 2017.5375457875457
Median: 2021.0
Mode: 2023


The three numbers tell different parts of the story:

- The **mode** is the year with the most new notable models. It falls right after ChatGPT launched in late 2022, when companies raced to release their own models.
- The **median** is the middle year: half of all notable models since the 1950s were released in that year or later. It's remarkably recent — most of AI's notable models were built in just the last few years.
- The **mean** is noticeably *earlier* than the median. That's because a small number of very old models, from the 1950s to the 1990s, pull the average backward. When data is lopsided like this, the median gives a better picture of the "typical" model.

## Checking the math "the hard way"

To make sure I understand what pandas is doing, I'll calculate the same three numbers using only built-in Python — no pandas, and no ready-made statistics functions.

First, I read the CSV file again, line by line, and collect each model's year into a list.

In [30]:
# "the hard way" structure with help from Claude
import csv
import io
import urllib.request

url = "https://epoch.ai/data/notable_ai_models.csv"

with urllib.request.urlopen(url) as response:
    text = response.read().decode("utf-8")

reader = csv.DictReader(io.StringIO(text))

years = []
for row in reader:
    date = row["Publication date"]
    if date != "":
        years.append(int(date[:4]))

print(len(years))

1092


**Mean:** add up every year, then divide by the number of models.

In [31]:
total = 0
for year in years:
    total = total + year

hard_mean = total / len(years)
print("Mean (the hard way):", hard_mean)

Mean (the hard way): 2017.5375457875457


**Median:** sort the years and take the middle one. If there's an even number of models, there are two middle years, so I average them.

In [32]:
sorted_years = sorted(years)
n = len(sorted_years)
middle = n // 2

if n % 2 == 0:
    hard_median = (sorted_years[middle - 1] + sorted_years[middle]) / 2
else:
    hard_median = sorted_years[middle]

print("Median (the hard way):", hard_median)

Median (the hard way): 2021.0


**Mode:** count how many models came out in each year using a dictionary, then find the year with the highest count.

In [33]:
year_counts = {}
for year in years:
    if year in year_counts:
        year_counts[year] = year_counts[year] + 1
    else:
        year_counts[year] = 1

hard_mode = None
highest_count = 0
for year in year_counts:
    if year_counts[year] > highest_count:
        highest_count = year_counts[year]
        hard_mode = year

print("Mode (the hard way):", hard_mode, "with", highest_count, "models")

Mode (the hard way): 2023 with 119 models


The results match pandas exactly. 

## Visualizing the boom

Below, each bar shows how many notable models were released in that year, scaled so the busiest year has the longest bar. Years before 2010 had very few models each, so I combine them into one line to keep the chart readable.

In [34]:
start_year = 2010
max_bar_length = 40

earlier_count = 0
for year in year_counts:
    if year < start_year:
        earlier_count = earlier_count + year_counts[year]

print("Notable AI models released per year")
print("(longest bar =", highest_count, "models)")
print()
print("Before", start_year, ":", earlier_count, "models in total")
print()

for year in sorted(year_counts):
    if year >= start_year:
        count = year_counts[year]
        bar_length = round(count / highest_count * max_bar_length)
        bar = "█" * bar_length
        print(year, bar, count)

Notable AI models released per year
(longest bar = 119 models)

Before 2010 : 126 models in total

2010 ███ 10
2011 ████ 11
2012 ████ 13
2013 ██████ 19
2014 ███████████ 33
2015 ██████████ 29
2016 ████████████████ 49
2017 ██████████████████ 55
2018 ████████████ 37
2019 ███████████████████████ 68
2020 █████████████████ 50
2021 ███████████████████████████ 79
2022 ███████████████████████████████ 91
2023 ████████████████████████████████████████ 119
2024 █████████████████████████████████ 99
2025 █████████████████████████████████████ 109
2026 ████████████████████████████████ 95


The chart makes the pattern clear. For decades, only a handful of notable models came out each year. Growth picked up in the 2010s, then accelerated sharply in the 2020s.

One surprise: the count doesn't rise every single year. For example, 2018 had fewer notable models than 2017. My guess is that 2017 was an unusually big year for breakthroughs — it's the year the "Transformer" architecture was introduced, which later became the foundation of models like ChatGPT. In 2018, researchers may have been busy learning and building on these new ideas before the next wave of models arrived. Progress in AI seems to come in waves rather than in a straight line.

Note that the most recent year only covers part of the year, so its bar is likely to grow once the year is complete.

In [35]:
recent_year = 2020

recent_total = 0
for year in year_counts:
    if year >= recent_year:
        recent_total = recent_total + year_counts[year]

recent_share = recent_total / len(years) * 100

print(f"Models released before {start_year}: {earlier_count}")
print(f"Models released in {hard_mode} alone: {highest_count}")
print(
    f"Share of all notable models released in {recent_year} or later: {recent_share:.0f}%"
)

Models released before 2010: 126
Models released in 2023 alone: 119
Share of all notable models released in 2020 or later: 59%


## Digging deeper
Looking more closely at the numbers, a few things stand out.

**One year rivals six decades.** The busiest year on its own produced almost as many notable models as every year before 2010 combined. AI research existed for a long time, but for most of that time it moved slowly.

**Most of AI's history is very recent.** More than half of all notable models in the dataset were released in 2020 or later. In other words, if you wanted to study the history of AI, most of the material would come from the last few years.

**Growth isn't smooth.** Besides the drop in 2018, there is another dip in 2020 — the first year of the COVID-19 pandemic. One possible explanation is that disruptions to research teams and to hardware supply chains slowed down new releases, but this data alone can't confirm that.

**The boom may be reaching a plateau.** This is the most surprising pattern. After the peak right after ChatGPT's launch, the number of notable models didn't keep rising. Instead, it levelled off at roughly a hundred models per year. I can think of a few possible reasons:

- **Building top models has become extremely expensive.** Training frontier AI now requires huge amounts of computing power, so fewer organizations can afford to compete.
- **Companies may be releasing fewer, bigger models** instead of many small ones.
- **"Notable" is a moving target.** As AI improves, a new model has to do more to stand out, so Epoch's bar for "notable" may effectively rise over time.

If the first explanation is right, it matters for policy: a market where only a few well-funded companies can build cutting-edge AI raises questions about competition and concentration of power. To test this idea, I looked at who has been building these models since 2020.

In [36]:
recent_models = models[models["Year"] >= recent_year]
print("Number of models since", recent_year, ":", len(recent_models))
recent_models["Organization"].value_counts().head(10)

Number of models since 2020 : 642


Organization
OpenAI             72
Google DeepMind    37
Alibaba            35
Meta AI            32
Anthropic          23
Google             20
DeepMind           15
Microsoft          13
Google Research    13
ByteDance          11
Name: count, dtype: int64

In [37]:
recent_models["Country (of organization)"].value_counts().head(10)

Country (of organization)
United States of America                                                      292
China                                                                         119
United States of America,United States of America                              41
United Kingdom                                                                 19
South Korea                                                                    16
China,China                                                                    13
United States of America,United States of America,United States of America     11
France                                                                          7
United States of America,France                                                 7
Germany                                                                         4
Name: count, dtype: int64

In [38]:
recent_models["Model accessibility"].value_counts()

Model accessibility
Open weights (unrestricted)      179
API access                       161
Unreleased                       151
Open weights (non-commercial)     54
Open weights (restricted use)     51
Hosted access (no API)            42
Limited access                     3
Name: count, dtype: int64

In [39]:
top10_share = (
    recent_models["Organization"].value_counts().head(10).sum()
    / len(recent_models)
    * 100
)

countries = recent_models["Country (of organization)"]
us_share = countries.str.contains("United States", na=False).mean() * 100
china_share = countries.str.contains("China", na=False).mean() * 100

access = recent_models["Model accessibility"]
open_share = access.str.startswith("Open weights", na=False).mean() * 100
unreleased_share = (access == "Unreleased").mean() * 100

print(f"Share of models built by the top 10 organizations: {top10_share:.0f}%")
print(f"Share involving a U.S. organization: {us_share:.0f}%")
print(f"Share involving a Chinese organization: {china_share:.0f}%")
print(f"Share released with open weights: {open_share:.0f}%")
print(f"Share never released: {unreleased_share:.0f}%")

Share of models built by the top 10 organizations: 42%
Share involving a U.S. organization: 66%
Share involving a Chinese organization: 25%
Share released with open weights: 44%
Share never released: 24%


## Who is building AI?

To test whether AI development is becoming concentrated, I looked only at the models released since 2020 and asked three questions: which organizations built them, where those organizations are based, and whether the models were made available to the public.

**A small group of companies builds a large share of notable AI.** Just ten organizations account for a large share of all notable models since 2020. OpenAI tops the list on its own, but Google's AI work is spread across several labs — Google DeepMind, Google, DeepMind, and Google Research. If you combine them, Google has built even more notable models than OpenAI. Almost all of the top ten are big tech companies or startups backed by them, which supports the idea that building frontier AI increasingly requires deep pockets.

**AI development is essentially a two-country race.** Organizations based in the United States are involved in about two thirds of all notable models since 2020, and Chinese organizations in about one in four. Together, the two countries account for the vast majority, while the next country on the list has only a small fraction of China's total. Chinese companies are not just participating; Alibaba ranks third among all organizations since 2020, ahead of Meta and Anthropic. For anyone following U.S.–China technology competition, this is a striking picture of how few players are really in the game.

**Open models are a major force.** Despite the concentration among big companies, a large share of recent notable models were released with open weights, meaning anyone can download and run them. In fact, "open weights (unrestricted)" is the single most common category. At the same time, about a quarter of notable models were never released at all, which means some of the most important AI systems are visible only through their developers' own reports. This tension between open and closed AI is at the center of current policy debates: open models spread access and competition, but they also make powerful tools harder to control.

**A note on messy data.** The country column lists one country per organization, so a model built jointly by two American labs appears as "United States of America, United States of America." To handle this, I counted a model as involving a country if that country appeared anywhere in the field. This means models built through international collaboration count toward more than one country.

## Conclusion

I started with a simple question: does the "AI boom" actually show up in the data? The answer is yes — but the story is more interesting than "more and more AI every year."

### Key takeaways

1. **AI's history is overwhelmingly recent.** The median notable model is only a few years old, and more than half of all notable models were released in 2020 or later. For most of its 70-year history, AI moved slowly.
2. **ChatGPT marked a turning point.** The busiest year in the entire dataset came right after ChatGPT's launch, when companies raced to release models of their own.
3. **Progress comes in waves.** The count dipped in 2018 and again in 2020, so growth has not been a smooth upward line.
4. **The boom may be levelling off.** Since its peak, the number of notable models has held steady at roughly a hundred per year instead of continuing to climb.
5. **A few players, mostly in two countries.** Since 2020, a small group of big tech companies — almost all based in the U.S. or China — has built a large share of notable models, though many of those models are released openly.

### Why this matters

The fourth takeaway is the one I find most important. If the plateau reflects how expensive it has become to build cutting-edge AI, then fewer organizations can realistically compete at the frontier. That shifts the key question from *how many* models are being built to **who can afford to build them**. For policymakers, a market dominated by a handful of well-funded companies raises real questions about competition, access, and concentration of power.

### What's next

This naturally leads to the money behind the boom: **who is paying for all of this?** In my next project, I plan to look at SEC Form D filings — the public records U.S. startups file when they raise private funding — to see how much capital is flowing into AI companies, and whether it is concentrated in a few large rounds.

### Caveats

- This dataset only includes models that Epoch AI considers "notable," so it doesn't represent all AI models ever built.
- What counts as "notable" is Epoch's judgment, and newer models may be easier to find and document than older ones.
- The current year is incomplete, so its count will likely rise.
- This analysis shows *when* models were released, not *why*. The explanations for the dips and the plateau are hypotheses that would need further research to confirm.
- Some organizations appear under several names (for example, Google's different labs), so organization counts may understate how concentrated AI development really is.

### Sources

- Data: Epoch AI, [Data on AI Models](https://epoch.ai/data/ai-models), used under CC BY 4.0.